# 07 · Tables

This notebook writes the tables of the paper: **Table 1**, **Table 2** and **Supplementary Tables 1 to 4**.

Nothing is estimated here. Every number was calculated by an earlier notebook and is read from a file.
This notebook selects the rows and columns of each table and prints the numbers in the format of the paper.

| | |
|---|---|
| Reads | `SUMMARY/outcomes_summary.csv`, written by notebook 05 (Table 2, Supplementary Tables 3 and 4); `SET_CHARACTERISTICS` and `SET_COUNTS`, written by notebook 02 (Table 1, Supplementary Table 1) |
| Writes | every table as a `.csv` file and as a markdown file (`.md`) in `TABLES`. The last section, **Files written**, lists them |
| Needs | no GPU; a few seconds |

Supplementary Table 2 describes the three foundation models. It is typed in as text and does not depend on any result.

## Outline

1. **How numbers are printed**: rounding, signs, confidence intervals.
2. **How a table is shown and written**: one table, three forms (notebook, `.csv`, `.md`).
3. **Results of the analysis**: the summary file and the names of the configurations.
4. **Table 1**, **Table 2**, **Supplementary Tables 1 to 4**, in the order of the paper. Each section starts with the title and legend of the table as in the paper.
5. **Files written**.

In [ ]:
%run ./00_setup_and_settings.ipynb

In [ ]:
import json

import numpy as np
import pandas as pd
from IPython.display import display

## How numbers are printed

**Decimals.** Proportions (decision change, direction agreement, individual decisions) are in percent with one decimal, and their differences are in percentage points with one decimal. AUROCs and their differences have three decimals.
In the two tables of the sets, ages are in whole years, percentages have one decimal, and intervals in hours have one decimal (Table 1) or two (Supplementary Table 1).

**Rounding.** One rule is used for every number of this notebook: a half is rounded away from zero (26.45 becomes 26.5, and −0.0125 becomes −0.013). Two details make this rule explicit in the code:

- Python's own `round` and its number formatting round a half to the even neighbour, and they work on the binary number that is stored, which is often slightly below the decimal number that was meant
  (2.675 is stored as 2.67499999… and `round(2.675, 2)` gives 2.67). `round_half_away` therefore does the rounding itself.
- A very small tolerance (0.00002 of the last printed digit) is added before the number is cut, so that a value that is a half up to the precision of the stored number is treated as a half.

**Signs.** A difference always carries its sign, `+` or `−` (the minus sign U+2212, not the hyphen). A difference that rounds to zero is printed without a sign (`0.0`, `0.000`).
Values (a baseline, an AUROC) are printed without a sign.

**Confidence intervals.** A 95% confidence interval follows its estimate in the same cell, in parentheses: `−11.0 (−12.5 to −9.4)`.
The confidence limits come from the bootstrap of notebook 05. They depend on the resamples that were drawn, so they can differ in the last digit from the limits printed in the paper. The estimates do not depend on the resamples.

In [ ]:
MINUS = "\u2212"                                     # the minus sign of the tables (U+2212)


def round_half_away(v, decimals=1):
    """Rounds v to the given number of decimals; a half is rounded away from zero."""
    v = float(v)
    r = np.floor(abs(v) * 10 ** decimals + 0.5 + 2e-5) / 10 ** decimals
    return -r if (v < 0 and r > 0) else r


def fmt(v, decimals=1):
    """A value without a sign, for example 24.3 or 0.924."""
    return f"{round_half_away(v, decimals):.{decimals}f}"


def fmt_signed(v, decimals=1):
    """A difference with its sign, for example −11.0 or +0.004. A difference that rounds to zero has no sign."""
    r = round_half_away(v, decimals)
    if abs(r) < 0.5 * 10 ** -decimals:
        return f"{0:.{decimals}f}"
    return ("+" if r > 0 else MINUS) + f"{abs(r):.{decimals}f}"


for _v, _d in ((26.45, 1), (-0.0125, 3), (0.0004, 3), (-0.04, 1), (2.675, 2)):
    print(f"{_v!r:>8} with {_d} decimal(s): value {fmt(abs(_v), _d):>6} | difference {fmt_signed(_v, _d):>6}")

## How a table is shown and written

Every table is built once, as a pandas DataFrame whose cells are the printed texts. From this one object the function `show_and_save` makes three forms:

- **In the notebook**: the DataFrame itself, with the text aligned to the left. Tables with grouped columns (for example one group of columns for each foundation model) have two levels of column names.
  Tables with blocks of rows (for example one block for each group of outcomes) have two levels of row names.
- **`.csv`**: the same cells with one header line. The name of a group of columns and the name of the column are joined with a colon (`ECGFounder: Baseline`), and the block of a row is the first column.
  The file is written as UTF-8 with a byte order mark, so that spreadsheet programs recognize the encoding (the cells contain the minus sign `−` and the dash `–`).
  The cells are texts as printed. For calculations use the numbers in `SUMMARY/outcomes_summary.csv`.
- **`.md`**: a plain markdown table in the layout of the paper.
  A markdown table has a single header line, so a table with grouped columns is written as follows: the header line has the name of each group in the first column of the group (the group extends to the next name),
  and the first line of the body has the names of the columns in bold. A block of rows starts with a line that has only the title of the block, in bold, as in the paper.

In [ ]:
def _md_line(cells):
    """One line of a markdown table."""
    return "|" + "|".join(f" {c} " if c != "" else " " for c in (str(c).replace("|", "\\|") for c in cells)) + "|"


def markdown_lines(table, block_rows=True):
    """Lines of a plain markdown table in the layout of the paper.
    table: DataFrame of texts. Two levels of columns = grouped columns. Two levels of rows = blocks of rows:
    with block_rows the title of a block is a line of its own, otherwise it is the first column and is written once (Table 1)."""
    n = table.shape[1]
    blocks = table.index.nlevels == 2
    lead = list(table.index.names) if (blocks and not block_rows) else [table.index.names[-1]]       # names of the leading column(s)
    if table.columns.nlevels == 2:
        groups = [g if (i == 0 or g != table.columns[i - 1][0]) else "" for i, (g, _) in enumerate(table.columns)]
        lines = [_md_line(lead + groups), _md_line(["---"] * (len(lead) + n)), _md_line([""] * len(lead) + [f"**{c}**" for _, c in table.columns])]
    else:
        lines = [_md_line(lead + list(table.columns)), _md_line(["---"] * (len(lead) + n))]
    previous = None
    for key, cells in zip(table.index, table.values.tolist()):
        if blocks and block_rows:
            if key[0] != previous:
                lines.append(_md_line([f"**{key[0]}**"] + [""] * n))
            lines.append(_md_line([key[1]] + cells))
        elif blocks:
            lines.append(_md_line([key[0] if key[0] != previous else "", key[1]] + cells))
        else:
            lines.append(_md_line([key] + cells))
        previous = key[0] if blocks else None
    return lines


def show_and_save(table, name, block_rows=True):
    """Shows a table in the notebook and writes it to TABLES as {name}.csv and {name}.md."""
    flat = table.copy()
    if flat.columns.nlevels == 2:
        flat.columns = [f"{group}: {column}" for group, column in flat.columns]
    flat.reset_index().to_csv(TABLES / f"{name}.csv", index=False, encoding="utf-8-sig")
    (TABLES / f"{name}.md").write_text("\n".join(markdown_lines(table, block_rows)) + "\n", encoding="utf-8")
    # in the notebook: text aligned to the left and to the top of the cell, so that long texts and blocks of rows read easily
    display(table.style.set_table_styles([{"selector": "th, td", "props": [("text-align", "left"), ("vertical-align", "top")]}]))
    print(f"written to TABLES: {name}.csv, {name}.md ({table.shape[0]} rows, {table.shape[1]} columns)")


def text_table(keys, cells, columns, index_names):
    """DataFrame of texts. keys: one name (or one pair block, name) per row; cells: one list of texts per row; columns: names or pairs (group, name)."""
    index = pd.MultiIndex.from_tuples(keys, names=index_names) if isinstance(keys[0], tuple) else pd.Index(keys, name=index_names)
    cols = pd.MultiIndex.from_tuples(columns) if isinstance(columns[0], tuple) else pd.Index(columns)
    return pd.DataFrame(cells, index=index, columns=cols)

## Results of the analysis

`outcomes_summary.csv` has one row for each configuration of each foundation model (mean of the three seeds). The last section of notebook 05 documents its columns. The tables use, for an outcome `x`:

| Column | Content |
|---|---|
| `x_value`, `x_value_lo`, `x_value_hi` | value of the outcome and its 95% confidence interval |
| `x_diff`, `x_diff_lo`, `x_diff_hi` | paired difference from the baseline of the same foundation model and its 95% confidence interval |

A configuration is identified by the columns `training`, `head` and `reading`. The two configurations that all tables share:

| Name in the paper | `training` | `head` | `reading` |
|---|---|---|---|
| Baseline | `baseline` | `model` | `single` |
| Three remedies (final setting) | `student` | `pen300` | `mixed0.9` |

The names `pen300` and `mixed0.9` are built from the settings `PENALTY_WEIGHT` and `MIX_SINGLE` of notebook 00 (the number in `mixed0.9` is the weight of the window average, 1 − `MIX_SINGLE`).

The file is read with `float_precision="round_trip"`, which returns exactly the numbers that were stored, so that the rounding of a printed value does not depend on the reader.

In [ ]:
outcomes = pd.read_csv(SUMMARY / "outcomes_summary.csv", float_precision="round_trip")

PENALIZED = f"pen{PENALTY_WEIGHT}"                       # "pen300": the penalized head of the final setting
MIXED = f"mixed{1 - MIX_SINGLE:g}"                       # "mixed0.9": the mixed reading of the final setting
BASELINE = ("baseline", "model", "single")               # training, head, reading
THREE_REMEDIES = ("student", PENALIZED, MIXED)


def configuration(model, training, head, reading):
    """The row of outcomes_summary.csv for one configuration of one foundation model."""
    q = outcomes[(outcomes["model"] == model) & (outcomes["training"] == training) & (outcomes["head"] == head) & (outcomes["reading"] == reading)]
    assert len(q) == 1, f"{len(q)} rows for {model}, {training}, {head}, {reading} in outcomes_summary.csv"
    return q.iloc[0]


def value_cell(row, outcome, decimals, interval=False):
    """Value of an outcome, with its 95% confidence interval if interval is set: "5.8 (5.4 to 6.1)"."""
    text = fmt(row[f"{outcome}_value"], decimals)
    if interval:
        text += f" ({fmt(row[f'{outcome}_value_lo'], decimals)} to {fmt(row[f'{outcome}_value_hi'], decimals)})"
    return text


def difference_cell(row, outcome, decimals):
    """Paired difference from the baseline with its 95% confidence interval: "−11.0 (−12.5 to −9.4)"."""
    return f"{fmt_signed(row[f'{outcome}_diff'], decimals)} ({fmt_signed(row[f'{outcome}_diff_lo'], decimals)} to {fmt_signed(row[f'{outcome}_diff_hi'], decimals)})"


baseline = {m: configuration(m, *BASELINE) for m in MODELS}
remedies = {m: configuration(m, *THREE_REMEDIES) for m in MODELS}
print(f"{len(outcomes)} configurations in outcomes_summary.csv | final setting: head {PENALIZED}, reading {MIXED}")

## Table 1

**Table 1 | Datasets and evaluation sets.** AUROC, area under the receiver operating characteristic curve; IQR, interquartile range.

The numbers of this table are the characteristics of the sets that notebook 02 wrote to `SET_CHARACTERISTICS`: recordings, patients and pairs, age, sex, the prevalence of the reference labels and the interval between the two recordings of a pair.
The other cells (use of a set, reference standard) are descriptions and are typed in below.

- Age and sex are summarized over recordings, so both recordings of a pair count.
- MIMIC-IV-ECG has neither age nor sex. Notebook 02 takes them from the patients table of MIMIC-IV when that table is present (`MIMIC_PATIENTS`, optional). Without it these cells show `NA`.
- The numbers are rounded with the rule of this notebook, a half away from zero. This matters for percentages that lie exactly between two printed values: 1,918 of 4,000 recordings from women are 47.95%, which prints as 48.0.

### Characteristics of the sets

`SET_CHARACTERISTICS` has one row for each set (the section **File formats** of notebook 02 documents its columns), and `SET_COUNTS` has the numbers of the flow diagram.
Both are written by the same run of notebook 02, so the sizes of the sets must agree between them; the cell below checks this before any table is built.

In [ ]:
if not (SET_CHARACTERISTICS.exists() and SET_COUNTS.exists()):
    raise FileNotFoundError("The files of the sets (SET_CHARACTERISTICS, SET_COUNTS) were not found. Run notebook 02_training_and_evaluation_sets first: it writes both files.")

characteristics = pd.read_csv(SET_CHARACTERISTICS, float_precision="round_trip").set_index("set")     # one row per set; exactly the stored numbers
set_counts = json.loads(SET_COUNTS.read_text(encoding="utf-8"))

# the two files must describe the same sets
for _key, _name in (("train", "ptbxl_train"), ("val", "ptbxl_val"), ("test", "ptbxl_test"), ("test_adjacent", "ptbxl_test_adjacent")):
    assert set_counts["ptbxl"][_key] == characteristics.loc[_name, "recordings"], _name
assert set_counts["mimic"]["pool"]["recordings"] == characteristics.loc["mimic_pool", "recordings"] and set_counts["mimic"]["pool"]["patients"] == characteristics.loc["mimic_pool", "patients"]
for _dataset in ("mimic", "code15", "echonext"):
    for _name, _c in set_counts[_dataset]["sets"].items():                 # one patient gives one recording, or one pair, to a set
        assert _c["selected"] == characteristics.loc[_name, "patients"], _name
print(f"{len(characteristics)} sets:", ", ".join(characteristics.index))


def count_text(n):
    """A count with thousands separators: 17,084."""
    return f"{int(n):,}"


def age_text(s):
    """Median age and quartiles of a set, "61 (50–72)", or NA if no age is known."""
    return f"{fmt(s['age_median'], 0)} ({fmt(s['age_q1'], 0)}–{fmt(s['age_q3'], 0)})" if s["age_n"] > 0 else "NA"


def women_text(s):
    """Percentage of women of a set, or NA if no sex is known."""
    return fmt(s["women_pct"], 1) if s["sex_n"] > 0 else "NA"

### The table

One row for each set, grouped by dataset. Sets of single recordings show recordings and patients, and sets of pairs show the number of pairs.
The column Reference standard is a description; where it contains a number of the set (median interval, prevalence of a label), the number is filled in from `SET_CHARACTERISTICS`.

In [ ]:
# dataset, set, use, name of the set in SET_CHARACTERISTICS, reference standard ({interval} = median interval in hours; {CD}, {NORM}, {HYP} = prevalence of the label in percent)
TABLE1_ROWS = [
    ("PTB-XL (Germany)", "Training", "Label training", "ptbxl_train", "Statements assigned by cardiologists, five diagnostic superclasses"),
    ("PTB-XL (Germany)", "Validation", "Checkpoint selection, decision thresholds", "ptbxl_val", ""),
    ("PTB-XL (Germany)", "Test", "AUROC", "ptbxl_test", ""),
    ("PTB-XL (Germany)", "Test, subset for the comparison of adjacent segments", "Decision change between adjacent segments", "ptbxl_test_adjacent", ""),
    ("MIMIC-IV-ECG (USA)", "Unlabelled training pool", "Distillation target, no labels used", "mimic_pool", "None"),
    ("MIMIC-IV-ECG (USA)", "Test", "Decision change between adjacent segments, AUROC", "mimic_test", "Machine interpretation statements mapped to the five superclasses"),
    ("MIMIC-IV-ECG (USA)", "Pairs, all", "Decision change between recordings", "mimic_pairs_all", "Interval 1 min to 24 h (median {interval} h)"),
    ("MIMIC-IV-ECG (USA)", "Pairs, identical machine report", "Decision change between recordings", "mimic_pairs_same_report", "Interval 1 min to 24 h (median {interval} h)"),
    ("CODE-15% (Brazil)", "Test", "Decision change between adjacent segments, AUROC", "code15_test", "Labels from physician reports: conduction disturbance ({CD}%), normal ECG ({NORM}%)"),
    ("CODE-15% (Brazil)", "Pairs with a change in conduction label", "Direction agreement", "code15_cd_change", "Conduction label present in one ECG of the pair"),
    ("CODE-15% (Brazil)", "Pairs with a change in normal label", "Direction agreement", "code15_normal_change", "Normal label present in one ECG of the pair"),
    ("EchoNext (USA)", "Test", "Decision change between adjacent segments, AUROC", "echonext_test", "Left ventricular wall thickness of 13 mm or more on echocardiography ({HYP}%)"),
]

_keys, _cells = [], []
for _dataset, _set, _use, _name, _reference in TABLE1_ROWS:
    s = characteristics.loc[_name]
    _size = f"{count_text(s['pairs'])} pairs" if pd.notna(s["pairs"]) else f"{count_text(s['recordings'])} ({count_text(s['patients'])})"      # sets of pairs are counted in pairs
    # the numbers of this set that a description can contain (a description that asks for a number that the set does not have raises a KeyError)
    _numbers = {k: fmt(s[column], 1) for k, column in (("interval", "interval_h_median"), ("CD", "prevalence_CD"), ("NORM", "prevalence_NORM"), ("HYP", "prevalence_HYP")) if pd.notna(s[column])}
    _keys.append((_dataset, _set)); _cells.append([_use, _size, age_text(s), women_text(s), _reference.format(**_numbers)])
table1 = text_table(_keys, _cells, ["Use", "Recordings (patients) or pairs", "Age, median (IQR), years", "Women (%)", "Reference standard"], ["Dataset", "Set"])
show_and_save(table1, "Table1", block_rows=False)

## Table 2

**Table 2 | Decision changes, discrimination and response to documented change in three foundation models at baseline, and differences with the three remedies.**
Decision change: the decision for at least one of the five diagnostic classes differed between the two readings. Values are means of three training seeds.
Differences are paired with the baseline of the same foundation model, with 95% bootstrap confidence intervals, and are in percentage points for decision change and for direction agreement.
Baseline: network fine-tuned on labels only, read once over the 5-s segment with its own output layer (model head).
Three remedies: distillation, mixed reading (weight of 0.9 for the window average) and penalized head (penalty weight 300).
Decision thresholds were set at 90% sensitivity in the PTB-XL validation fold. AUROC, area under the receiver operating characteristic curve; CI, confidence interval.

For each foundation model the table has two columns: the value of the baseline (`x_value` in the row of the baseline) and the difference of the three remedies from the baseline with its confidence interval
(`x_diff`, `x_diff_lo`, `x_diff_hi` in the row of the three remedies). The rows are the outcomes, in four groups.

In [ ]:
# title of the group, decimals, [(outcome in outcomes_summary.csv, name of the row)]
TABLE2_ROWS = [
    ("Decision change between adjacent segments, % of recordings", 1,
     [("adjacent_ptbxl", "PTB-XL"), ("adjacent_mimic", "MIMIC-IV-ECG"), ("adjacent_code15", "CODE-15%"), ("adjacent_echonext", "EchoNext")]),
    ("Decision change between recordings, % of pairs", 1,
     [("between_same_report", "MIMIC-IV-ECG, identical machine report"), ("between_all", "MIMIC-IV-ECG, all pairs")]),
    ("Discrimination, AUROC", 3,
     [("auroc_ptbxl", "PTB-XL, five diagnostic classes"), ("auroc_mimic", "MIMIC-IV-ECG, five diagnostic classes"), ("auroc_code15_cd", "CODE-15%, conduction disturbance"),
      ("auroc_code15_normal", "CODE-15%, normal ECG"), ("auroc_echonext", "EchoNext, wall thickness of 13 mm or more")]),
    ("Direction agreement in pairs with a documented change, % of pairs", 1,
     [("direction_cd", "CODE-15%, conduction disturbance"), ("direction_normal", "CODE-15%, normal ECG")]),
]

_keys, _cells = [], []
for _title, _decimals, _rows in TABLE2_ROWS:
    for _outcome, _name in _rows:
        _keys.append((_title, _name))
        _cells.append([c for m in MODELS for c in (value_cell(baseline[m], _outcome, _decimals), difference_cell(remedies[m], _outcome, _decimals))])
table2 = text_table(_keys, _cells, [(MODEL_NAMES[m], c) for m in MODELS for c in ("Baseline", "Difference (95% CI)")], ["Group of outcomes", "Outcome"])
show_and_save(table2, "Table2")

## Supplementary Table 1

**Supplementary Table 1 | Training and evaluation sets.** Age and sex are summarized over recordings. Ages above 89 years were set to 90 for the summary (PTB-XL records them as 300, and MIMIC-IV gives an anchor age of 91).
Age and sex of the patients in MIMIC-IV-ECG were taken from the patients table of MIMIC-IV, and age at the recording was calculated from the anchor age and the anchor year.
IQR, interquartile range; NORM, normal ECG; MI, myocardial infarction; STTC, ST/T change; CD, conduction disturbance; HYP, hypertrophy.

The same file as for Table 1 (`SET_CHARACTERISTICS`, read above), with one row for each set and more detail: recordings and patients in separate columns, and in the last column

- the prevalence of every reference label that the dataset has, in percent of the recordings (CODE-15%: conduction disturbance and normal ECG; EchoNext: wall thickness of 13 mm or more);
- for the pairs of MIMIC-IV-ECG, the interval between the two recordings (median, quartiles, and the percentage of pairs with an interval of at most 1 h).

The sets of pairs have two recordings for each pair, so their number of recordings is twice their number of patients.
As in Table 1, age and sex of the MIMIC-IV-ECG sets show `NA` if the patients table of MIMIC-IV was not available to notebook 02.
The last line of the output says for which part of the MIMIC-IV-ECG recordings age and sex were found in that table.

In [ ]:
# name of the set in SET_CHARACTERISTICS -> name of the row
SUPP1_ROWS = {
    "ptbxl_train": "PTB-XL training (folds 1-8)", "ptbxl_val": "PTB-XL validation (fold 9)", "ptbxl_test": "PTB-XL test (fold 10)", "ptbxl_test_adjacent": "PTB-XL test, subset for adjacent segments",
    "mimic_pool": "MIMIC-IV-ECG unlabelled pool", "mimic_test": "MIMIC-IV-ECG test", "mimic_pairs_all": "MIMIC-IV-ECG pairs, all", "mimic_pairs_same_report": "MIMIC-IV-ECG pairs, identical machine report",
    "code15_test": "CODE-15% test", "code15_cd_change": "CODE-15% conduction-change pairs", "code15_normal_change": "CODE-15% normal-change pairs", "echonext_test": "EchoNext test",
}


def labels_or_interval(s):
    """Last column of Supplementary Table 1 for one set."""
    if pd.notna(s["interval_h_median"]):                                # pairs of MIMIC-IV-ECG
        return f"interval, h: median {fmt(s['interval_h_median'], 2)} (IQR {fmt(s['interval_h_q1'], 2)}-{fmt(s['interval_h_q3'], 2)}); within 1 h {fmt(s['interval_within_1h_pct'], 1)}%"
    if s["dataset"] == "code15":                                        # two labels; empty for the sets of pairs
        return f"CD {fmt(s['prevalence_CD'], 1)}; NORM {fmt(s['prevalence_NORM'], 1)}" if pd.notna(s["prevalence_CD"]) else ""
    if s["dataset"] == "echonext":                                      # the label is the wall thickness on echocardiography
        return f"wall thickness of 13 mm or more {fmt(s['prevalence_HYP'], 1)}"
    return "; ".join(f"{c} {fmt(s[f'prevalence_{c}'], 1)}" for c in CLASSES)


assert list(characteristics.index) == list(SUPP1_ROWS), "the sets of SET_CHARACTERISTICS are not those of Supplementary Table 1"
_cells = [[count_text(s["recordings"]), count_text(s["patients"]), age_text(s), women_text(s), labels_or_interval(s)] for _, s in characteristics.iterrows()]
supp1 = text_table(list(SUPP1_ROWS.values()), _cells, ["Recordings", "Patients", "Age, median (IQR)", "Women, %", "Label prevalence (%) or pair interval"], "Set")
show_and_save(supp1, "SupplementaryTable1")
# age and sex of MIMIC-IV-ECG are known only for the patients who have a row in the patients table of MIMIC-IV
_mimic = characteristics[characteristics["dataset"] == "mimic"]
print(f"MIMIC-IV-ECG: age known for {fmt(100 * _mimic['age_n'].sum() / _mimic['recordings'].sum(), 1)}% and sex for {fmt(100 * _mimic['sex_n'].sum() / _mimic['recordings'].sum(), 1)}% of the recordings")

## Supplementary Table 2

**Supplementary Table 2 | Foundation models and settings of their adaptation.** Parameters are those of the network as fine-tuned with five outputs.
The learning rate is that of the network; the output layer was trained with 1 x 10⁻³ in every model.
All models were trained for 3,000 steps with the same schedule, batch size, loss and rule for the selection of checkpoints (Supplementary Methods).
CODE-15% is a subset of CODE. PTB-XL is part of the PhysioNet/Computing in Cardiology Challenge 2021 collection. EchoNext was not used in the pretraining of any model.

This table is a description and is not calculated from any file. The settings in the columns Input and Adaptation are those that notebooks 00 and 03 implement
(input conventions of the three models, trained layers, learning rates).

In [ ]:
SUPP2_COLUMNS = ["Architecture", "Pretraining", "Pretraining data", "Evaluation data present in pretraining", "Input", "Adaptation"]
SUPP2_ROWS = {
    "ECGFounder": [
        "Convolutional network (Net1D) with 12 leads as input channels; 30.8 million parameters; embedding of 1,024 dimensions",
        "Supervised, 150 diagnostic labels",
        "10.8 million ECGs of 1.8 million patients (Harvard-Emory ECG Database)",
        "None",
        "500 Hz, standardized over all leads and time within the input",
        "All layers trained; learning rate 1 x 10⁻⁴"],
    "HuBERT-ECG (base size)": [
        "Convolutional feature extractor and 12 transformer layers, with the leads concatenated in time; 92.8 million parameters; embedding of 768 dimensions",
        "Self-supervised, masked prediction of cluster labels",
        "9.1 million ECGs from four countries",
        "PTB-XL, MIMIC-IV-ECG and CODE, without labels",
        "100 Hz, each lead scaled to the range −1 to 1 within the input",
        "Transformer layers trained, feature extractor fixed; learning rate 1 x 10⁻⁵"],
    "ECG-FM": [
        "Convolutional feature extractor and 12 transformer layers, with 12 leads as input channels; 90.4 million parameters; embedding of 768 dimensions",
        "Self-supervised, masked reconstruction combined with contrastive learning",
        "Public checkpoint pretrained on MIMIC-IV-ECG and the PhysioNet/Computing in Cardiology Challenge 2021 collection",
        "PTB-XL and MIMIC-IV-ECG, without labels",
        "500 Hz, each lead standardized within the input",
        "Transformer layers trained, feature extractor fixed; learning rate 1 x 10⁻⁵"],
}
supp2 = text_table(list(SUPP2_ROWS), list(SUPP2_ROWS.values()), SUPP2_COLUMNS, "Model")
show_and_save(supp2, "SupplementaryTable2")

## Supplementary Table 3

**Supplementary Table 3 | Individual decisions and diagnostic classes in three foundation models.**
Individual decisions changed: mean over the five diagnostic classes of the proportion of binary decisions that differed between the two readings; by diagnostic class, the proportion for that class.
Values in parentheses are 95% bootstrap confidence intervals. Differences are paired with the baseline of the same model and are absolute: percentage points for the proportions of decisions changed.
Values were calculated separately for each of three seeds and then averaged; the outputs of the models were not averaged. AUROC, area under the receiver operating characteristic curve.

For each foundation model the table has three columns: baseline, three remedies, and their paired difference. The rows are in four blocks:

| Block | Outcomes in `outcomes_summary.csv` | Cells |
|---|---|---|
| Individual decisions changed | `individual_adjacent_{dataset}`, `individual_between_same_report`, `individual_between_all` | values and difference, each with its confidence interval |
| Decisions changed between adjacent segments in PTB-XL, by class | `adjacent_ptbxl_{CLASS}` | values and difference, each with its confidence interval |
| AUROC in PTB-XL, by class | `auroc_ptbxl_{CLASS}` | values; difference with its confidence interval |
| AUROC in MIMIC-IV-ECG, by class | `auroc_mimic_{CLASS}` | values; difference with its confidence interval |

In [ ]:
# title of the block, decimals, confidence interval of the values?, [(outcome in outcomes_summary.csv, name of the row)]
SUPP3_ROWS = [
    ("Individual decisions changed, % (mean over the five diagnostic classes)", 1, True,
     [("individual_adjacent_ptbxl", "Adjacent segments, PTB-XL"), ("individual_adjacent_mimic", "Adjacent segments, MIMIC-IV-ECG"), ("individual_adjacent_code15", "Adjacent segments, CODE-15%"),
      ("individual_adjacent_echonext", "Adjacent segments, EchoNext"), ("individual_between_same_report", "Between recordings, identical machine report"), ("individual_between_all", "Between recordings, all pairs")]),
    ("Decisions changed between adjacent segments in PTB-XL by diagnostic class, %", 1, True, [(f"adjacent_ptbxl_{c}", CLASS_NAMES[c]) for c in CLASSES]),
    ("AUROC in PTB-XL by diagnostic class", 3, False, [(f"auroc_ptbxl_{c}", CLASS_NAMES[c]) for c in CLASSES]),
    ("AUROC in MIMIC-IV-ECG by diagnostic class", 3, False, [(f"auroc_mimic_{c}", CLASS_NAMES[c]) for c in CLASSES]),
]

_keys, _cells = [], []
for _title, _decimals, _interval, _rows in SUPP3_ROWS:
    for _outcome, _name in _rows:
        _keys.append((_title, _name))
        _cells.append([c for m in MODELS for c in (value_cell(baseline[m], _outcome, _decimals, _interval), value_cell(remedies[m], _outcome, _decimals, _interval),
                                                   difference_cell(remedies[m], _outcome, _decimals))])
supp3 = text_table(_keys, _cells, [(MODEL_NAMES[m], c) for m in MODELS for c in ("Baseline", "Three remedies", "Difference")], ["Group of measures", "Measure"])
show_and_save(supp3, "SupplementaryTable3")

## Supplementary Table 4

**Supplementary Table 4 | All outcomes for every setting in three foundation models.**
The row of the baseline shows the value, and all other rows show the paired difference from the baseline of the same model with its 95% bootstrap confidence interval in parentheses.
Differences are absolute: percentage points for decision change and for direction agreement.
Decision change: proportion of recordings or pairs in which the decision for at least one of the five diagnostic classes differed.
Baseline: network fine-tuned on labels only, read once with its own output layer (model head). The second to fourth rows add the remedies in turn, and the fourth row is the final setting (penalty weight 300).
The row + Distillation is also distillation alone. In the rows for the penalty weight, only the head of the distilled model with mixed reading was changed.
The rows for training on 2.5-s windows, the direct agreement loss and self-distillation (model head, single read) are given for ECGFounder, in which the training methods were compared (Supplementary Fig. 2a).
Values were calculated separately for each of three seeds and then averaged; the outputs of the models were not averaged. AUROC, area under the receiver operating characteristic curve; CD, conduction disturbance.

The rows are the settings below, repeated for each foundation model. The columns are the thirteen outcomes of Table 2, in four groups.

| Row | `training` | `head` | `reading` |
|---|---|---|---|
| Baseline | `baseline` | `model` | `single` |
| + Distillation | `student` | `model` | `single` |
| + Mixed reading | `student` | `model` | `mixed0.9` |
| + Penalized head (three remedies) | `student` | `pen300` | `mixed0.9` |
| Mixed reading alone | `baseline` | `model` | `mixed0.9` |
| Penalized head alone | `baseline` | `pen300` | `single` |
| Mixed reading and penalized head, no retraining | `baseline` | `pen300` | `mixed0.9` |
| Distillation and penalized head, single read | `student` | `pen300` | `single` |
| Penalty weight 0 (refit only), 30, 100, 1,000 | `student` | `pen0`, `pen30`, `pen100`, `pen1000` | `mixed0.9` |
| Training on 2.5-s windows; Direct agreement loss; Self-distillation (ECGFounder only) | `windows`; `agreement`; `selfdistill` | `model` | `single` |

The rows for the penalty weight are the weights of `PENALTY_WEIGHTS` other than the weight of the final setting, which is the fourth row.

In [ ]:
# name of the row, training, head, reading, foundation models that have the setting
SETTINGS = [
    ("Baseline", *BASELINE, MODELS),
    ("+ Distillation", "student", "model", "single", MODELS),
    ("+ Mixed reading", "student", "model", MIXED, MODELS),
    ("+ Penalized head (three remedies)", *THREE_REMEDIES, MODELS),
    ("Mixed reading alone", "baseline", "model", MIXED, MODELS),
    ("Penalized head alone", "baseline", PENALIZED, "single", MODELS),
    ("Mixed reading and penalized head, no retraining", "baseline", PENALIZED, MIXED, MODELS),
    ("Distillation and penalized head, single read", "student", PENALIZED, "single", MODELS),
]
SETTINGS += [(f"Penalty weight {g:,}" + (" (refit only)" if g == 0 else ""), "student", f"pen{g}", MIXED, MODELS) for g in PENALTY_WEIGHTS if g != PENALTY_WEIGHT]
SETTINGS += [("Training on 2.5-s windows", "windows", "model", "single", ["ecgfounder"]),               # the training methods were compared in ECGFounder only
             ("Direct agreement loss", "agreement", "model", "single", ["ecgfounder"]),
             ("Self-distillation", "selfdistill", "model", "single", ["ecgfounder"])]

# group of columns, decimals, [(outcome in outcomes_summary.csv, name of the column)]
SUPP4_COLUMNS = [
    ("Decision change between adjacent segments, %", 1, [("adjacent_ptbxl", "PTB-XL"), ("adjacent_mimic", "MIMIC-IV-ECG"), ("adjacent_code15", "CODE-15%"), ("adjacent_echonext", "EchoNext")]),
    ("Decision change between recordings, %", 1, [("between_same_report", "Identical report"), ("between_all", "All pairs")]),
    ("AUROC", 3, [("auroc_ptbxl", "PTB-XL"), ("auroc_mimic", "MIMIC-IV-ECG"), ("auroc_code15_cd", "CODE-15%, CD"), ("auroc_code15_normal", "CODE-15%, normal"), ("auroc_echonext", "EchoNext")]),
    ("Direction agreement, %", 1, [("direction_cd", "CD"), ("direction_normal", "Normal")]),
]
_columns = [(group, outcome, name, decimals) for group, decimals, members in SUPP4_COLUMNS for outcome, name in members]

_keys, _cells = [], []
for m in MODELS:
    for _name, _training, _head, _reading, _models in SETTINGS:
        if m not in _models:
            continue
        row = configuration(m, _training, _head, _reading)
        _keys.append((MODEL_NAMES[m], _name))
        # the baseline has no difference from itself: its row shows the values
        _cells.append([value_cell(row, o, d) if (_training, _head, _reading) == BASELINE else difference_cell(row, o, d) for _, o, _, d in _columns])
supp4 = text_table(_keys, _cells, [(group, name) for group, _, name, _ in _columns], ["Model", "Setting"])
show_and_save(supp4, "SupplementaryTable4")

## Files written

All files are in `TABLES` (`MASTER/output/tables`).

| Table of the paper | Files | Source of the numbers |
|---|---|---|
| Table 1 | `Table1.csv`, `Table1.md` | `SET_CHARACTERISTICS` (notebook 02) |
| Table 2 | `Table2.csv`, `Table2.md` | `outcomes_summary.csv` (notebook 05) |
| Supplementary Table 1 | `SupplementaryTable1.csv`, `SupplementaryTable1.md` | `SET_CHARACTERISTICS` (notebook 02) |
| Supplementary Table 2 | `SupplementaryTable2.csv`, `SupplementaryTable2.md` | typed in this notebook |
| Supplementary Table 3 | `SupplementaryTable3.csv`, `SupplementaryTable3.md` | `outcomes_summary.csv` (notebook 05) |
| Supplementary Table 4 | `SupplementaryTable4.csv`, `SupplementaryTable4.md` | `outcomes_summary.csv` (notebook 05) |

The `.csv` files have one header line and are encoded as UTF-8 with a byte order mark (`pd.read_csv(file)` reads them directly). The `.md` files are plain markdown tables in the layout of the paper.
The cells of both are the printed texts, with the minus sign `−` (U+2212) in negative numbers.

In [ ]:
for _f in sorted(TABLES.glob("*Table*.*")):
    print(f"{_f.name:28s} {_f.stat().st_size:7,d} bytes")